In [1]:
# 1. INSTALAÇÃO DE DEPENDÊNCIAS
!pip install gradio -q
!pip install demucs -q
!pip install stable-ts -q
!pip install openai -q
!pip install pydub -q
!pip install coqui-tts -q

# 2. IMPORTS E CONFIGURAÇÃO
import gradio as gr
import os
import shutil
import subprocess
import stable_whisper
from openai import OpenAI
import torch
from pydub import AudioSegment # Para manipulação de áudio
import math

# !! IMPORTANTE: Resolve a necessidade de digitar 'y' para os termos da Coqui TTS !!
# Esta linha DEVE vir ANTES de importar a biblioteca TTS.
os.environ['COQUI_TOS_AGREED'] = '1'

from TTS.api import TTS

print("✅ Ambiente final pronto com todas as dependências!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.3/185.3 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 64.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 57.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 119.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 18.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4

In [17]:
!pip install coqui-tts -q
import torch
from pydub import AudioSegment # Para manipulação de áudio
import math

# !! IMPORTANTE: Resolve a necessidade de digitar 'y' para os termos da Coqui TTS !!
# Esta linha DEVE vir ANTES de importar a biblioteca TTS.
os.environ['COQUI_TOS_AGREED'] = '1'

from TTS.api import TTS

In [10]:
# 3. CARREGAMENTO DOS MODELOS DE IA (WHISPER E TTS)
print("Carregando modelos de IA... (Isso pode levar vários minutos)")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Carrega Whisper
print("-> Carregando Stable Whisper...")
whisper_model = stable_whisper.load_model('medium')
print("✅ Modelo Whisper carregado.")

# Carrega Coqui TTS
print("-> Carregando Coqui XTTS...")
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ Modelo Coqui TTS carregado.")

# 4. FUNÇÃO AUXILIAR PARA MUDAR VELOCIDADE DO ÁUDIO
def speed_change(sound, speed=1.0):
    sound_with_altered_frame_rate = sound._spawn(sound.raw_data, overrides={
        "frame_rate": int(sound.frame_rate * speed)
    })
    return sound_with_altered_frame_rate.set_frame_rate(sound.frame_rate)

print("\n🚀 Modelos e funções auxiliares prontos para uso!")

Carregando o modelo Whisper... (Isso pode levar alguns minutos na primeira vez)


100%|█████████████████████████████████████| 1.42G/1.42G [00:19<00:00, 79.0MiB/s]


✅ Modelo Whisper carregado com sucesso!


In [18]:
print("Carregando modelos de IA... (Isso pode levar vários minutos)")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Carrega Coqui TTS
print("-> Carregando Coqui XTTS...")
tts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ Modelo Coqui TTS carregado.")

Carregando modelos de IA... (Isso pode levar vários minutos)
-> Carregando Coqui XTTS...


100%|█████████▉| 1.86G/1.87G [00:31<00:00, 105MiB/s]
100%|██████████| 1.87G/1.87G [00:31<00:00, 59.6MiB/s]
4.37kiB [00:00, 37.5kiB/s]

361kiB [00:00, 485kiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 184iB/s]
100%|██████████| 7.75M/7.75M [00:13<00:00, 19.6MiB/s]

✅ Modelo Coqui TTS carregado.


In [19]:
# 4. FUNÇÃO AUXILIAR PARA MUDAR VELOCIDADE DO ÁUDIO
def speed_change(sound, speed=1.0):
    sound_with_altered_frame_rate = sound._spawn(sound.raw_data, overrides={
        "frame_rate": int(sound.frame_rate * speed)
    })
    return sound_with_altered_frame_rate.set_frame_rate(sound.frame_rate)

print("\n🚀 Modelos e funções auxiliares prontos para uso!")


🚀 Modelos e funções auxiliares prontos para uso!


In [28]:
# 5. A GRANDE FUNÇÃO FINAL E COMPLETA
def gerar_versao_traduzida(audio_filepath, idioma_original, idioma_alvo, openai_api_key, ajuste_volume_db, progress=gr.Progress(track_tqdm=True)):
    """
    Executa o pipeline COMPLETO: Separa -> Transcreve -> Traduz -> Gera Voz -> Monta a música final.
    """
    try:
        # ETAPA 1: SEPARAÇÃO (DEMUCS)
        progress(0.05, desc="[1/6] Separando faixas com Demucs...")
        output_dir = "/content/separated"
        nome_base_arquivo = os.path.splitext(os.path.basename(audio_filepath))[0]
        if os.path.exists(output_dir): shutil.rmtree(output_dir)
        command = f'python3 -m demucs --two-stems vocals -o "{output_dir}" "{audio_filepath}"'
        subprocess.run(command, shell=True, check=True, capture_output=True, text=True)
        caminho_resultado_demucs = os.path.join(output_dir, "htdemucs_ft", nome_base_arquivo)
        if not os.path.exists(caminho_resultado_demucs): caminho_resultado_demucs = os.path.join(output_dir, "htdemucs", nome_base_arquivo)
        caminho_vocal, caminho_instrumental = os.path.join(caminho_resultado_demucs, "vocals.wav"), os.path.join(caminho_resultado_demucs, "no_vocals.wav")

        # ETAPA 2: TRANSCRIÇÃO (WHISPER)
        progress(0.20, desc="[2/6] Transcrevendo a letra do vocal...")
        resultado_whisper = whisper_model.transcribe(caminho_vocal, language=idioma_original, regroup=True)
        segmentos_originais = resultado_whisper.segments

        # ETAPA 3: TRADUÇÃO (OPENAI)
        progress(0.35, desc="[3/6] Traduzindo a letra com OpenAI...")
        if not openai_api_key: raise gr.Error("Chave da API da OpenAI não fornecida.")
        client = OpenAI(api_key=openai_api_key)
        letra_completa_numerada = "".join([f"{i}: {seg.text.strip()}\n" for i, seg in enumerate(segmentos_originais)])
        prompt_sistema = f"""
Você é um tradutor especialista em letras de música, traduzindo do {idioma_original} para o {idioma_alvo}.
Sua tarefa é traduzir a letra a seguir.

Regras importantes:
1.  **Contexto é tudo**: Não traduza literalmente. Capture a emoção, a poesia e o significado da música.
2.  **Métrica e Ritmo (Regra Adicionada)**: Este é um ponto crucial. Esforce-se para que a contagem de sílabas fonéticas de cada verso em português seja o mais próxima possível da contagem do verso original em inglês. O objetivo é criar uma versão que possa ser cantada, mantendo o fluxo e a cadência da melodia original.
3.  **Consistência no Refrão**: Versos que se repetem (como refrões ou pontes) DEVEM ser traduzidos exatamente da mesma forma todas as vezes que aparecerem.
4.  **Formato de Saída**: A letra está numerada. Retorne a tradução mantendo EXATAMENTE a mesma numeração linha por linha. Não adicione ou remova linhas. O formato deve ser:
    NÚMERO: Tradução da linha

Exemplo de saída esperada:
0: Tradução da primeira linha
1: Tradução da segunda linha
2: Tradução da terceira linha
...
"""
        resposta_openai = client.chat.completions.create(model="gpt-4o", messages=[{"role": "system", "content": prompt_sistema}, {"role": "user", "content": letra_completa_numerada}], temperature=0.3)
        traducao_bruta = resposta_openai.choices[0].message.content
        traducoes_mapeadas = {int(p[0]): p[1].strip() for ln in traducao_bruta.strip().split('\n') if len(p := ln.split(':', 1)) == 2}
        segmentos_traduzidos = [{"inicio": s.start, "fim": s.end, "texto_traduzido": traducoes_mapeadas.get(i, "")} for i, s in enumerate(segmentos_originais)]

        # ETAPA 4: GERAÇÃO E ALINHAMENTO DA VOZ (TTS)
        progress(0.50, desc="[4/6] Gerando e alinhando nova voz (TTS)...")
        duracao_total_ms = math.ceil(segmentos_traduzidos[-1]['fim']) * 1000
        faixa_vocal_final_tts = AudioSegment.silent(duration=duracao_total_ms)
        pasta_segmentos_temp = "segmentos_tts"
        os.makedirs(pasta_segmentos_temp, exist_ok=True)

        for i, seg in enumerate(progress.tqdm(segmentos_traduzidos, desc="Processando segmentos TTS")):
            duracao_original_s = seg['fim'] - seg['inicio']
            if duracao_original_s < 0.2 or not seg['texto_traduzido']: continue

            caminho_temp = os.path.join(pasta_segmentos_temp, f"temp_{i}.wav")
            tts_model.tts_to_file(text=seg['texto_traduzido'], speaker_wav=caminho_vocal, language=idioma_alvo, file_path=caminho_temp)

            audio_gerado = AudioSegment.from_wav(caminho_temp)
            if len(audio_gerado) == 0: continue

            duracao_gerada_s = len(audio_gerado) / 1000.0
            fator_velocidade = duracao_gerada_s / duracao_original_s

            # =================== CORREÇÃO IMPORTANTE AQUI ===================
            # Restaurando a sua lógica original que funciona corretamente.
            if fator_velocidade == 0: continue # Evita divisão por zero se o áudio for vazio

            if fator_velocidade > 1.0:
                # Áudio gerado é MAIS LONGO que o original -> PRECISA ACELERAR
                audio_alinhado = audio_gerado.speedup(playback_speed=fator_velocidade)
            else:
                # Áudio gerado é MAIS CURTO que o original -> PRECISA DESACELERAR
                # A função speed_change espera um fator < 1.0 para desacelerar
                audio_alinhado = speed_change(audio_gerado, fator_velocidade)
            # ================================================================

            faixa_vocal_final_tts = faixa_vocal_final_tts.overlay(audio_alinhado, position=seg['inicio'] * 1000)

        caminho_vocal_traduzido = "/content/vocal_final_traduzido_e_alinhado.wav"
        faixa_vocal_final_tts.export(caminho_vocal_traduzido, format="wav")
        print("Nova faixa vocal gerada!")

        # ETAPA 5: JUNÇÃO FINAL (VOCAL TTS + INSTRUMENTAL)
        progress(0.85, desc="[5/6] Mixando a música final...")
        instrumental = AudioSegment.from_wav(caminho_instrumental)
        vocal_gerado = AudioSegment.from_wav(caminho_vocal_traduzido)

        vocal_ajustado = vocal_gerado + ajuste_volume_db
        musica_final = instrumental.overlay(vocal_ajustado)

        caminho_musica_final = "musica_final_completa.mp3"
        musica_final.export(caminho_musica_final, format="mp3", bitrate="192k")
        print(f"Música final montada e salva em {caminho_musica_final}!")

        # ETAPA 6: RETORNO PARA A INTERFACE
        progress(1.0, desc="[6/6] Concluído!")
        # Retorna os dois áudios que queremos exibir na interface
        return caminho_vocal_traduzido, caminho_musica_final

    except Exception as e:
        import traceback
        traceback.print_exc()
        raise gr.Error(f"Ocorreu um erro: {e}")

In [ ]:
# 6. INTERFACE FINAL E COMPLETA
lista_idiomas_codigos = ['en', 'pt', 'es', 'fr', 'de', 'ja', 'ru', 'it', 'ko', 'zh']

interface = gr.Interface(
    fn=gerar_versao_traduzida,
    inputs=[
        gr.Audio(type="filepath", label="1. Envie sua Música (MP3 ou WAV)"),
        gr.Dropdown(lista_idiomas_codigos, value='en', label="2. Idioma Original da Música"),
        gr.Dropdown(lista_idiomas_codigos, value='pt', label="3. Idioma da Tradução"),
        gr.Textbox(label="4. Sua Chave da API da OpenAI", type="password", placeholder="Cole sua chave 'sk-...' aqui..."),
        gr.Slider(-12, 12, value=0, step=1, label="5. Ajuste de Volume do Vocal (em dB)", info="Aumente para o vocal ficar mais alto, diminua para ficar mais baixo.")
    ],
    outputs=[
        gr.Audio(label="Voz Gerada (TTS)"),
        gr.Audio(label="Música Final (Voz Gerada + Instrumental)")
    ],
    title="Estúdio de Adaptação Musical com IA 🤖🎤",
    description="PROJETO COMPLETO: A IA separa o vocal, transcreve, traduz, clona a voz do cantor no novo idioma e monta a versão final da música.",
    allow_flagging="never"
)

interface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a0d958934a1be35d83.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Nova faixa vocal gerada!
Música final montada e salva em musica_final_completa.mp3!
